# Workload mix and the March 2025 split

Monthly GPU-enabled versus non-GPU containers, and the percentile values before and after March 2025, the first month from which GPU-enabled containers outnumber non-GPU containers in every month (Sections 3.2 and 4.3).

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()

from cohort_contract import current_contract
LATEST_CONTRACT=current_contract()


## 1. Inputs and checks

In [ ]:
# Inputs and outputs stay inside the repository.
from pathlib import Path
from IPython.display import display as _cleanup_display, HTML as _CleanupHTML

RESULTS = RESULTS_REVISION
RESULTS.mkdir(parents=True, exist_ok=True)

def display(*objects, **kwargs):
    """Display-only terminology; dataframe values and calculation keys are unchanged."""
    for obj in objects:
        html = obj.data if isinstance(obj, _CleanupHTML) else (
            obj._repr_html_() if hasattr(obj, "_repr_html_") else None
        )
        if isinstance(html, str):
            html = html.replace("GPU-Enabled", "Integer-GPU")
            _cleanup_display(_CleanupHTML(html), **kwargs)
        else:
            _cleanup_display(obj, **kwargs)

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display as _cleanup_display

ROOT = Path(str(PROCESSED_DATA))
USAGE = ROOT / "UsageLog20230401_20251130_20260301_20260831.csv"
TRACE = ROOT / "TimeSeries_20240926_20251130_20260301_20260831.parquet"
BOUNDARY = pd.Timestamp("2025-03-01")
FROZEN = {("Non-GPU", "Pre"): (90, 85),
          ("GPU-Enabled", "Pre"): (90, 119),  # GPU Pre 119 (Train P90 119.0)
          ("Non-GPU", "Post"): (70, 146),
          ("GPU-Enabled", "Post"): (70, 170)}
for path in [USAGE, TRACE]:
    if not path.is_file():
        raise FileNotFoundError(path)

def normalized_filename(values):
    return (values.astype("string").str.replace("\\", "/", regex=False)
            .str.split("/").str[-1].str.strip()
            .str.replace(r"\.(csv|parquet)$", "", regex=True, case=False))

usage = pd.read_csv(USAGE, usecols=["filename", "created_time",
                                    "gpu_allocated(group)"], low_memory=False)
usage["filename_clean"] = normalized_filename(usage["filename"])
usage["created_time"] = pd.to_datetime(usage["created_time"], errors="raise")
usage = usage.loc[usage["created_time"].ge("2023-03-01")
                  & usage["created_time"].lt("2025-12-01")].copy()
if usage["filename_clean"].isna().any() or usage["filename_clean"].duplicated().any():
    raise ValueError("UsageLog Train 키 결측·중복")
lookup = usage.set_index("filename_clean")
parquet = pq.ParquetFile(TRACE)
if parquet.metadata.num_row_groups != LATEST_CONTRACT['row_groups']:
    raise ValueError("원본 Parquet row group 수가 기존 입력과 다름")

group_info = []
for group_id in range(parquet.metadata.num_row_groups):
    rg = parquet.metadata.row_group(group_id)
    name_stat = rg.column(8).statistics
    gpu_stat = rg.column(6).statistics
    if name_stat is None or gpu_stat is None or name_stat.min != name_stat.max or gpu_stat.min != gpu_stat.max:
        raise ValueError(f"row group {group_id}: 단일 컨테이너/GPU 그룹 보장 안 됨")
    key = str(name_stat.min).replace("\\", "/").split("/")[-1].strip()
    if key.endswith((".csv", ".parquet")):
        key = key.rsplit(".", 1)[0]
    if key in lookup.index:
        record = lookup.loc[key]
        if str(gpu_stat.min) != record["gpu_allocated(group)"]:
            raise ValueError(f"GPU 그룹 불일치: {key}")
        if str(gpu_stat.min) not in {"0.0_GPU", "1.0_GPU", "2.0_GPU", "3.0_GPU", "4.0_GPU"}:
            raise ValueError(f"예상 밖 GPU 그룹: {gpu_stat.min}")
        group_info.append((group_id, key, record["created_time"],
                           str(gpu_stat.min), rg.num_rows))
meta = pd.DataFrame(group_info, columns=["group_id", "filename_clean",
                                         "created_time", "gpu_group", "sample_rows"])
if len(meta) != LATEST_CONTRACT['population_counts']['Train'] or meta["filename_clean"].duplicated().any() or int(meta.sample_rows.sum()) != LATEST_CONTRACT['train_rows']:
    raise ValueError("기존 Train population/행 수 불일치")
meta["cohort"] = np.where(meta.created_time.lt(BOUNDARY), "3월 이전 / Pre-Mar",
                          "3월 이후 / Mar+")
meta["gpu_bin"] = np.where(meta.gpu_group.eq("0.0_GPU"), "비GPU / Non-GPU",
                           "GPU 사용 / GPU-Enabled")
mix = (meta.groupby(["cohort", "gpu_bin"], observed=True)
       .size().unstack(fill_value=0)
       .reindex(["3월 이전 / Pre-Mar", "3월 이후 / Mar+"], fill_value=0))
mix["전체 / Total"] = mix.sum(axis=1)
mix["GPU 사용 비율 / GPU share (%)"] = mix["GPU 사용 / GPU-Enabled"] / mix["전체 / Total"] * 100
print("[RESULT] population_and_workload_mix: PASS")
mix_display = mix.rename(columns={
    "GPU 사용 비율 / GPU share (%)": "정수 GPU 비율 / Integer-GPU share (%)"
})
display(mix_display)
print("Train containers:", len(meta), "TimeSeries rows:", int(meta.sample_rows.sum()))

# Figure 3의 전체 제출 집단과 IR2 적격 집단을 혼동하지 않도록 월별 분모를 대조한다.
source_month = usage.loc[usage.created_time.ge("2024-09-01")].copy()
source_month["월 / Month"] = source_month.created_time.dt.to_period("M").astype(str)
eligible_month = meta.loc[meta.created_time.ge("2024-09-01")].copy()
eligible_month["월 / Month"] = eligible_month.created_time.dt.to_period("M").astype(str)
monthly_rows = []
for month in sorted(source_month["월 / Month"].unique()):
    full = source_month.loc[source_month["월 / Month"].eq(month)]
    eligible = eligible_month.loc[eligible_month["월 / Month"].eq(month)]
    full_gpu = int(full["gpu_allocated(group)"].ne("0.0_GPU").sum())
    eligible_gpu = int(eligible.gpu_group.ne("0.0_GPU").sum())
    monthly_rows.append({"월 / Month": month,
        "전체 로그 GPU / Full-log GPU": full_gpu,
        "전체 로그 비GPU / Full-log non-GPU": len(full) - full_gpu,
        "전체 로그 fractional / Full-log fractional": int(full["gpu_allocated(group)"].eq("0.x_GPU").sum()),
        "적격 GPU / Eligible GPU": eligible_gpu,
        "적격 비GPU / Eligible non-GPU": len(eligible) - eligible_gpu})
monthly_mix = pd.DataFrame(monthly_rows).set_index("월 / Month")
if monthly_mix[["전체 로그 GPU / Full-log GPU", "전체 로그 비GPU / Full-log non-GPU"]].sum().sum() != len(source_month):
    raise ValueError("월별 전체 UsageLog 합계 오류")
if monthly_mix[["적격 GPU / Eligible GPU", "적격 비GPU / Eligible non-GPU"]].sum().sum() != len(eligible_month):
    raise ValueError("월별 적격 Train 합계 오류")
print("[RESULT] full_log_vs_monitored_mix: PASS")
monthly_display = monthly_mix.rename(columns={
    "전체 로그 GPU / Full-log GPU": "전체 로그 GPU 유형 / Full-log GPU-enabled",
    "적격 GPU / Eligible GPU": "적격 정수 GPU / Eligible Integer-GPU",
})
display(monthly_display)

# Cohort and applied IR2 timeouts (85/119/146/170) from the frozen cohort contract.
sys.path.insert(0, str(PROJECT_ROOT / "04_Analysis/revision"))
from cohort_contract import current_contract
LATEST_CONTRACT = current_contract()


## 2. 기준 산출 사례와 유휴시간을 독립 재계산

In [ ]:
summary_rows = []
needed = ["time", "used_time(minute)", "cpu_util(%)", "mem_util(%)", "gpu_util(%)"]
for record in meta.itertuples(index=False):
    block = parquet.read_row_group(record.group_id, columns=needed).to_pandas()
    if len(block) != record.sample_rows or block[["time","used_time(minute)","cpu_util(%)","mem_util(%)"] + (["gpu_util(%)"] if record.gpu_group != "0.0_GPU" else [])].isna().any().any():
        raise ValueError(f"행 수·필수 결측: {record.filename_clean}")
    block = block.sort_values("used_time(minute)", kind="mergesort")
    if record.gpu_group == "0.0_GPU":
        idle = block["cpu_util(%)"].le(10) & block["mem_util(%)"].le(15)
    else:
        idle = (block["cpu_util(%)"].le(10) & block["mem_util(%)"].le(20)
                & block["gpu_util(%)"].eq(0))
    active = ~idle
    start = float(block["used_time(minute)"].iloc[0])
    end = float(block["used_time(minute)"].iloc[-1])
    first_time = block["time"].min()
    if first_time.tzinfo is not None:
        raise ValueError('V2 requires naive KST timestamps; unexpected timezone')
    first_kst = first_time
    start_gap = (first_kst - record.created_time).total_seconds() / 60
    if bool(active.any()):
        active_time = block.loc[active, "used_time(minute)"]
        pre = max(float(active_time.iloc[0]) - start, 0)
        post = max(end - float(active_time.iloc[-1]), 0)
        valid_pre = pre >= 1
        valid_post = post >= 1 and bool(idle.iloc[-1])
    else:
        pre = post = np.nan
        valid_pre = valid_post = False
    summary_rows.append((record.filename_clean, record.cohort, record.gpu_bin,
                         bool(active.any()), valid_pre, valid_post,
                         pre, post, start_gap))
container = pd.DataFrame(summary_rows, columns=[
    "filename_clean", "cohort", "gpu_bin", "normal",
    "valid_pre", "valid_post", "pre_min", "post_min", "start_gap_min"])
check = (container.groupby("gpu_bin")[["normal", "valid_pre", "valid_post"]]
         .sum().reindex(["비GPU / Non-GPU", "GPU 사용 / GPU-Enabled"]))
# Empty metric values are zero (rebuild_time_series.py).
expected = pd.DataFrame([[3784, 2921, 2588], [2940, 2761, 1849]],
                        index=check.index, columns=check.columns)
if not check.equals(expected):
    raise ValueError(f"01번 사례 수 재현 실패\n{check}\n{expected}")
if container.filename_clean.duplicated().any() or len(container) != len(meta):
    raise ValueError("컨테이너 중복·누락")
print("[RESULT] baseline_phase_reproduction: PASS")
display(check.rename(columns={"normal": "Normal / 활동 경험",
                              "valid_pre": "Valid Pre / 유효 Pre",
                              "valid_post": "Valid Post / 유효 Post"}))


### 결과
## 3. 전후 어느 시기가 기준을 더 많이 구성했는가?

In [ ]:
reference_raw = {("비GPU / Non-GPU", "Pre"): 85.0,
                 ("GPU 사용 / GPU-Enabled", "Pre"): 119.0,  # applied 119
                 ("비GPU / Non-GPU", "Post"): 145.9,
                 ("GPU 사용 / GPU-Enabled", "Post"): 169.6}
result_rows = []
for gpu_label, gpu_key in [("비GPU / Non-GPU", "Non-GPU"),
                           ("GPU 사용 / GPU-Enabled", "GPU-Enabled")]:
    for phase in ["Pre", "Post"]:
        percentile, timeout = FROZEN[(gpu_key, phase)]
        case_col, value_col = (("valid_pre", "pre_min") if phase == "Pre"
                               else ("valid_post", "post_min"))
        full = container.loc[container.gpu_bin.eq(gpu_label)
                             & container[case_col], value_col].to_numpy(float)
        periods = {}
        for cohort in ["3월 이전 / Pre-Mar", "3월 이후 / Mar+"]:
            vals = container.loc[container.gpu_bin.eq(gpu_label)
                                 & container.cohort.eq(cohort)
                                 & container[case_col], value_col].to_numpy(float)
            periods[cohort] = vals
        pre = periods["3월 이전 / Pre-Mar"]
        post = periods["3월 이후 / Mar+"]
        if len(pre) + len(post) != len(full):
            raise ValueError("사례 수 합계 불일치")
        p_pre, p_post, p_all = [float(np.percentile(a, percentile))
                                for a in [pre, post, full]]
        if not np.isclose(p_all, reference_raw[(gpu_label, phase)], atol=.01, rtol=0):
            raise ValueError(f"01번 전체 원시 분위수 불일치: {gpu_label}/{phase}: {p_all}")
        over_pre, over_post, over_all = [int(np.count_nonzero(a > timeout))
                                         for a in [pre, post, full]]
        if over_pre + over_post != over_all:
            raise ValueError("초과 건수 합계 불일치")
        result_rows.append({
            "GPU 유형 / GPU type": gpu_label, "단계 / Phase": phase,
            "기준 산출 분위수 / Percentile": percentile,
            "이전 유효 사례 / Pre cases": len(pre),
            "이후 유효 사례 / Post cases": len(post),
            "이후 사례 비중 / Post share (%)": len(post) / len(full) * 100,
            "이전 분위수 / Pre quantile (min)": p_pre,
            "이후 분위수 / Post quantile (min)": p_post,
            "전체 분위수 / All quantile (min)": p_all,
            "원고 고정 기준 / Frozen cutoff (min)": timeout,
            "이전 초과 / Pre exceed (%)": over_pre / len(pre) * 100,
            "이후 초과 / Post exceed (%)": over_post / len(post) * 100,
            "전체 초과 / All exceed (%)": over_all / len(full) * 100,
            "초과율 변화 / Difference (pp)": (over_post / len(post) - over_pre / len(pre)) * 100,
            "이전 초과 건수 / Pre exceed count": over_pre,
            "이후 초과 건수 / Post exceed count": over_post,
        })
threshold_table = pd.DataFrame(result_rows).set_index(
    ["GPU 유형 / GPU type", "단계 / Phase"])
print("[RESULT] threshold_source_and_stability: PASS")
display(threshold_table.round(2))


### 결과
## 4. 생성일과 첫 관측 사이 공백 확인

In [ ]:
gap_rows = []
for gpu_label in ["비GPU / Non-GPU", "GPU 사용 / GPU-Enabled"]:
    for cohort in ["3월 이전 / Pre-Mar", "3월 이후 / Mar+"]:
        part = container.loc[container.gpu_bin.eq(gpu_label)
                             & container.cohort.eq(cohort) & container.valid_pre]
        near = part.loc[part.start_gap_min.le(10), "pre_min"]
        gap_rows.append({
            "GPU 유형 / GPU type": gpu_label,
            "시기 / Cohort": cohort,
            "유효 Pre / Valid Pre": len(part),
            "첫 관측 >10분 / Start gap >10 min": int(part.start_gap_min.gt(10).sum()),
            "첫 관측 >10분 비율 / Share (%)": part.start_gap_min.gt(10).mean() * 100,
            "10분 이내 사례 / Near-start cases": len(near),
            "10분 이내 Pre P90 / Near-start P90 (min)": np.percentile(near, 90) if len(near) else np.nan,
        })
gap_table = pd.DataFrame(gap_rows).set_index(["GPU 유형 / GPU type", "시기 / Cohort"])
if (gap_table["첫 관측 >10분 / Start gap >10 min"]
    + gap_table["10분 이내 사례 / Near-start cases"]).ne(
        gap_table["유효 Pre / Valid Pre"]).any():
    raise ValueError("시작 공백 집단 합계 오류")
print("[RESULT] pre_start_coverage_sensitivity: PASS")
display(gap_table.round(2))


### 결과

In [ ]:
# Deterministic exports of existing retained summary tables (no new analysis).
mix.to_csv(RESULTS / "09_before_after_mix.csv", index=True)
monthly_mix.to_csv(RESULTS / "09_monthly_workload_mix.csv", index=True)
threshold_table.to_csv(RESULTS / "09_threshold_case_composition.csv", index=True)
gap_table.to_csv(RESULTS / "09_observation_start_gap.csv", index=True)

container.to_csv(RESULTS/"09_container_cases.csv",index=False)
